In [3]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_processing.ukhls_variables as _ukhls_vars
importlib.reload(_ukhls_vars)

import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from data_processing.ukhls_variables import (
    VARIABLE_MAP,
    CLUSTER_VARS,
    CATEGORICAL_VARS,
    CATEGORY_MAPS,
    SUMMARY_VARS,
    VARIABLES,
)

# ── Config ────────────────────────────────────────────────────────────────────
ZONE            = "E01003555"
K_CLUSTERS      = 5
WAVE            = "o"

SYNPOP_PARQUET  = "../data/7_synthetic_population/synthetic_population.parquet"
BACKFILL_PKL    = "../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"
REAL_PKL        = "../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"
OUTPUT_DIR      = "../data/8_regional_cluster"

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, f"{ZONE}_dna.csv")

# ── Load zone ─────────────────────────────────────────────────────────────────
print(f"Loading synthetic population for zone {ZONE} ...")
df_zone = pd.read_parquet(SYNPOP_PARQUET, filters=[("synthetic_zone", "==", ZONE)])
print(f"  {len(df_zone):,} persons in zone {ZONE}")

if len(df_zone) == 0:
    raise ValueError(f"No rows found for zone {ZONE} — check the zone code.")

# ── One-hot encode jbstat ─────────────────────────────────────────────────────
jbstat_col  = f"{WAVE}_jbstat"
jbstat_cats = CATEGORY_MAPS.get("jbstat", {})

df_backfill = pd.read_pickle(BACKFILL_PKL)

# Summary-only vars: in SUMMARY_VARS, not in CLUSTER_VARS, not jbstat/alljbstat,
# and present in the backfill pickle (not already in the feature-eng pickle).
alljbstat_codes    = [k for k in SUMMARY_VARS if k.startswith("alljbstat")]
summary_only_codes = [
    k for k in SUMMARY_VARS
    if k not in CLUSTER_VARS
    and k != "jbstat"
    and not k.startswith("alljbstat")
    and f"{WAVE}_{k}" in df_backfill.columns
]
backfill_cols = list(dict.fromkeys(
    ["pidp", jbstat_col] + [f"{WAVE}_{k}" for k in summary_only_codes]
))
jbstat_raw = df_backfill[backfill_cols].copy()

df_backfill_ohe = df_backfill[["pidp", jbstat_col]].copy()
df_backfill_ohe[jbstat_col] = df_backfill_ohe[jbstat_col].map(jbstat_cats).fillna("Unknown")
ohe = pd.get_dummies(df_backfill_ohe, columns=[jbstat_col], prefix="jbstat")
ohe_cols = [c for c in ohe.columns if c != "pidp"]
ohe[ohe_cols] = StandardScaler().fit_transform(ohe[ohe_cols].fillna(0))
df_zone = df_zone.merge(ohe, on="pidp", how="left")

# ── GMM ───────────────────────────────────────────────────────────────────────
feature_cols = [c for c in df_zone.columns if c not in ("synthetic_zone", "pidp")]
df_zone[feature_cols] = df_zone[feature_cols].fillna(0)
k = min(K_CLUSTERS, len(df_zone))
gm = GaussianMixture(n_components=k, covariance_type="full", n_init=10, random_state=42)
df_zone = df_zone.copy()
df_zone["tribe_id"] = gm.fit_predict(df_zone[feature_cols])
print(f"  GMM converged: {gm.converged_}  |  BIC: {gm.bic(df_zone[feature_cols]):,.0f}")

# ── Load real values for DNA ───────────────────────────────────────────────────
df_real = pd.read_pickle(REAL_PKL)
df_profiling = (
    df_zone[["pidp", "tribe_id"]]
    .merge(df_real, on="pidp", how="left")
    .merge(jbstat_raw, on="pidp", how="left")
)

# Derive binary alljbstat columns from raw o_jbstat
jbstat_numeric = pd.to_numeric(df_profiling[jbstat_col], errors="coerce")
for code in alljbstat_codes:
    num = int(code[len("alljbstat"):])
    df_profiling[f"{WAVE}_{code}"] = (jbstat_numeric == num).astype(float)

# ── Build DNA rows ────────────────────────────────────────────────────────────
def get_mode(series):
    m = series.dropna().mode()
    return float(m.iloc[0]) if not m.empty else np.nan

# All codes to profile: cluster vars + alljbstat + summary-only (e.g. oprlg1)
dna_codes = list(dict.fromkeys(
    CLUSTER_VARS
    + [k for k in alljbstat_codes if k not in CLUSTER_VARS]
    + [k for k in summary_only_codes if k not in CLUSTER_VARS]
))

dna_rows = []
for tribe_id in sorted(df_profiling["tribe_id"].unique()):
    tribe = df_profiling[df_profiling["tribe_id"] == tribe_id]
    row = {"tribe_id": tribe_id, "size": len(tribe)}
    jbstat_series = pd.to_numeric(tribe[jbstat_col], errors="coerce")
    mode_val = get_mode(jbstat_series)
    row["Employment status"] = jbstat_cats.get(mode_val, str(mode_val))
    for base_code in dna_codes:
        col = f"{WAVE}_{base_code}"
        if col not in tribe.columns:
            continue
        label = VARIABLE_MAP.get(base_code, base_code)
        series = pd.to_numeric(tribe[col], errors="coerce")
        if base_code.startswith("alljbstat"):
            row[label] = f"{round(series.mean() * 100)}%"
        elif base_code in CATEGORICAL_VARS and base_code in CATEGORY_MAPS:
            mode_val = get_mode(series)
            row[label] = CATEGORY_MAPS[base_code].get(mode_val, str(mode_val))
        else:
            row[label] = round(series.mean(), 1)
    dna_rows.append(row)

# ── Display & save ────────────────────────────────────────────────────────────
dna = pd.DataFrame(dna_rows).sort_values("size", ascending=False).reset_index(drop=True)
dna.to_csv(OUTPUT_CSV, index=False)
print(f"Saved DNA table to {OUTPUT_CSV}\n")

summary_labels = [VARIABLE_MAP[k] for k in SUMMARY_VARS if k in VARIABLE_MAP]
display_cols = ["tribe_id", "size"] + [l for l in summary_labels if l in dna.columns]
dna_summary = dna[display_cols].copy()

num_cols = dna_summary.select_dtypes(include="number").columns
dna_summary[num_cols] = dna_summary[num_cols].round(0).astype(object)

dna_summary.index = [
    f"Tribe {r['tribe_id']} ({r['Employment status']}, n={r['size']})"
    for _, r in dna_summary.iterrows()
]
dna_summary = dna_summary.drop(columns=["tribe_id", "size"])
print(dna_summary.to_string())


Loading synthetic population for zone E01003555 ...
  1,444 persons in zone E01003555
  GMM converged: True  |  BIC: -129,481
Saved DNA table to ../data/8_regional_cluster/E01003555_dna.csv

                                  Derived age at interview      Religion Highest qualification Monthly net pay (take-home) Social class (NS-SEC 8) Household size Number of children in household Mental health score (SF-12 MCS) Physical health score (SF-12 PCS) Minutes spent travelling to work Miles driven in last 12 months  Employment status Works at home Drives to work Employed Unemployed Retired Full-time student LT sick/disabled
Tribe 2 (Employed, n=707)                             42.0  Muslim/Islam                   2.0                      2113.0                     5.0            4.0                             1.0                            47.0                              52.0                             25.0                         2507.0           Employed            No             No   